# 06b · GSE65391 · RNA_array · do module scores follow disease activity within a child?

Reads `projected_scores.rds` (notebook 05b). Writes `data/run_artifacts/GSE65391/modules_across_visits.csv`.

**Samples.** All SLE samples: 924 samples from 158 children, the first visits and every later visit,
scored on the first-visit modules (notebook 05b).

**Why a mixed model.** Visits of the same child are not independent, so a correlation over all 924
samples would count each child many times. A linear mixed model with a random intercept for each
child accounts for this (Pinheiro JC, Bates DM. *Mixed-Effects Models in S and S-PLUS*. Springer,
2000; R package `nlme`).

**Within and between.** SLEDAI is split into two parts (Neuhaus JM, Kalbfleisch JD. Between- and
within-cluster covariate effects in the analysis of clustered data. *Biometrics* 1998;54:638-645):
- `sledai_between`: the child's mean SLEDAI over all their visits;
- `sledai_within`: the visit's SLEDAI minus that child's mean.

The within coefficient answers "when this child's disease is more active, does the module score
rise?". The between coefficient answers "do children with more active disease on average have higher
scores?". Children with one visit contribute to the between part only.

**Model, for each module:** `score ~ sledai_within + sledai_between`, random intercept per child,
REML (`nlme::lme`).

**Test.** The within-child coefficient, t and p for each module; BH q across the 14 modules.
**A finding** is q < 0.05.

**Comparison with Banchereau et al. 2016 (claim B3).** The paper reports that "the plasmablast
signature was the most robust biomarker of disease activity" in longitudinal data. If this holds
here, the plasma-cell module (pink) should have one of the strongest within-child
associations.

## Build the table of module scores and SLEDAI

In [1]:
source("../src/paths.R")
suppressMessages(library(nlme))
p <- readRDS(art("GSE65391", "projected_scores.rds"))
m <- p$meta; S <- p$scores
sle <- rownames(m)[m$disease == "SLE" & !is.na(m$sledai)]
d <- data.frame(subject = m[sle, "subject"], sledai = m[sle, "sledai"], S[sle, ])
d$sledai_between <- ave(d$sledai, d$subject)
d$sledai_within  <- d$sledai - d$sledai_between
c(samples = nrow(d), children = length(unique(d$subject)),
  children_with_2_or_more_visits = sum(table(d$subject) >= 2))

samples                       children 
                           924                            158 
children_with_2_or_more_visits 
                           131

**Explanation**
- `library(nlme)` loads the package for linear mixed models (Pinheiro J, Bates D, R Core Team. nlme: linear
  and nonlinear mixed effects models, R package).
- `projected_scores.rds` (notebook 05b) holds `scores`, every sample's 16 module scores, and `meta`, the
  sample table in the same order.
- `sle` holds the SLE samples with a recorded SLEDAI.
- `d` is one row per sample: the child, the SLEDAI and the 14 module scores. `S[sle, ]` adds one column per
  module, named by module.
- `ave(d$sledai, d$subject)` gives each sample its child's mean SLEDAI over all visits: `sledai_between`.
  For example, a child with SLEDAI 4, 8 and 12 gets 8 on all three samples.
- `sledai_within` is the visit's SLEDAI minus that mean. In the example: -4, 0 and 4.
- `sum(table(d$subject) >= 2)` counts the children with at least two samples.

**Result.** 924 SLE samples from 158 children; 131 children have two or more visits.

## Mixed model for each module

In [2]:
modules <- colnames(S)
fit <- do.call(rbind, lapply(modules, function(mo) {
  f <- lme(as.formula(paste(mo, "~ sledai_within + sledai_between")), random = ~ 1 | subject, data = d, method = "REML")
  tt <- summary(f)$tTable
  data.frame(module = mo,
             within_coef = tt["sledai_within", "Value"], within_t = tt["sledai_within", "t-value"], within_p = tt["sledai_within", "p-value"],
             between_coef = tt["sledai_between", "Value"], between_t = tt["sledai_between", "t-value"], between_p = tt["sledai_between", "p-value"])
}))
fit$within_q  <- p.adjust(fit$within_p, "BH")
fit$between_q <- p.adjust(fit$between_p, "BH")
fit <- fit[order(-abs(fit$within_t)), ]
format(fit[, c("module", "within_coef", "within_t", "within_q", "between_coef", "between_t", "between_q")], digits = 3)

,module,within_coef,within_t,within_q,between_coef,between_t,between_q
,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>
12,pink,0.003721,10.248,4.92e-22,0.003589,4.404,0.000163
9,black,0.003391,7.997,3.31e-14,0.004267,4.262,0.000163
5,salmon,-0.002767,-6.304,2.29e-09,-0.001940,-2.114,0.050510
2,cyan,0.001641,4.509,2.63e-05,0.004302,4.262,0.000163
8,tan,-0.001979,-3.975,2.16e-04,-0.001082,-1.383,0.196787
6,brown,-0.001693,-3.550,9.53e-04,0.002417,2.134,0.050510
14,purple,0.001599,2.755,1.20e-02,0.001073,1.735,0.107732
3,blue,0.001048,2.271,4.10e-02,0.002206,2.246,0.045723
11,green,0.000872,2.030,6.65e-02,-0.002962,-2.811,0.015595


**Explanation**
- `lapply(modules, function(mo) ...)` fits one model per module; `do.call(rbind, ...)` stacks the 16 result
  rows into one table.
- `as.formula(paste(mo, "~ sledai_within + sledai_between"))` builds the model formula from text. For
  example, for the black module: `black ~ sledai_within + sledai_between`.
- `lme(formula, random = ~ 1 | subject, data = d, method = "REML")` fits the model: the module score depends
  on the within-child and between-child parts of SLEDAI, and each child has its own baseline (a random
  intercept, `~ 1 | subject`). REML is the standard fitting method for such models.
- `summary(f)$tTable` is the table of coefficients: `Value`, `t-value` and `p-value` for each term.
- `p.adjust(p, "BH")` gives Benjamini-Hochberg q values across the 14 modules (Benjamini Y, Hochberg Y.
  *J R Stat Soc B* 1995;57:289-300).
- `order(-abs(fit$within_t))` sorts the modules from the largest to the smallest within-child t, ignoring
  its sign.
- `format(..., digits = 3)` prints the numbers with 3 significant digits.

**Result.** Within children, 8 of 14 modules change with SLEDAI at q < 0.05. The strongest are pink
(plasma cell, t = 10.2, q = 5e-22), black (interferon, t = 8.0), salmon (monocyte, falling, t = -6.3),
cyan (neutrophil granule, t = 4.5), tan (falling, t = -4.0) and brown (red cell, falling, t = -3.6).
Between children, the strongest are pink (t = 4.4), black and cyan (t = 4.3 each) and red (t = -3.1).

**Comparison with claim B3.** Rank of the plasma-cell module (pink) among the 14 modules by the
size of its within-child t statistic.

In [3]:
c(plasma_cell_rank_within = which(fit$module == "pink"), modules = nrow(fit))   # pink: the plasma-cell module (notebook 05)
fit[fit$module %in% c("pink", "black", "cyan", "blue"), c("module", "within_t", "within_q", "between_t", "between_q")]

plasma_cell_rank_within                 modules 
                      1                      14

,module,within_t,within_q,between_t,between_q
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
12,pink,10.247855,4.920865e-22,4.403642,0.000163345
9,black,7.996565,3.305970e-14,4.261558,0.000163345
2,cyan,4.509424,2.631907e-05,4.262472,0.000163345
3,blue,2.270784,4.101515e-02,2.245709,0.045723054


**Explanation**
- `which(fit$module == "pink")` gives the row of the plasma-cell module in the sorted table, which is
  its rank. For example, 1 means the strongest within-child association.
- The second line shows four modules side by side: plasma cell, interferon, neutrophil granule and blue.

**Result: claim B3 reproduced.** The plasma-cell module has the strongest within-child association
with disease activity of all 14 modules: when a child's SLEDAI rises, the plasma-cell score rises.
This is the paper's finding that the plasmablast signature is the most robust marker of disease
activity. Cross-sectionally, at first visits only, interferon was slightly stronger (notebook 06); the
longitudinal design reverses that order, as the paper's design would predict.

## Save

In [4]:
write.csv(fit, art("GSE65391", "modules_across_visits.csv"), row.names = FALSE)

**Explanation**
- `modules_across_visits.csv` keeps the model results of all 14 modules for later notebooks.